## Computing Chalcopyrite defect formation energies

In [5]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [6]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from matchest.aiida_utils.pmg import load_mp_struct
from ase.symbols import Formula
from aiida.engine import submit

from matchest.aiida_utils.workflows.simple_vac import SimpleVacancyWorkChain

In [7]:
basepath = GroupPathX('hc-defect')
workpath = basepath['workflows']
elemental_struct_path = GroupPathX('defects/elemental_ref')

In [8]:
elemental_struct_path.show_tree()

elemental_ref
├── As_bulk *
├── Au_bulk *
├── Ba_bulk *
├── Bi_bulk *
├── Ca_bulk *
├── Cd_bulk *
├── Cu_bulk *
├── Ge_bulk *
├── Hg_bulk *
├── In_bulk *
├── Mg_bulk *
├── O_bulk *
├── P_bulk *
├── Pb_bulk *
├── S_bulk *
├── Sb_bulk *
├── Se_bulk *
├── Sn_bulk *
├── Sr_bulk *
├── Te_bulk *
├── Tl_bulk *
└── Zn_bulk *



## Define the input to the wokchain

Generate structure

In [9]:
form = Formula('InCuSe2')  # Change the formula here
symbols = list(form)
ASite = symbols[0]
BSite = symbols[1]
CSite = symbols[2]

structure = load_mp_struct('mp-14090')

ps = structure.get_pymatgen()
ps['Tl'] = 'He'
ps['Cu'] = 'Ne' 
ps['Se'] = 'Xe'

ps['He'] = ASite  
ps['Ne'] = BSite 
ps['Xe'] = CSite

structure = orm.StructureData(pymatgen=ps)
print(ps)

Full Formula (In2 Cu2 Se4)
Reduced Formula: InCuSe2
abc   :   5.887256   5.887381   7.244696
angles: 113.984386 113.971472  89.992190
pbc   :       True       True       True
Sites (8)
  #  SP            a         b          c
---  ----  ---------  --------  ---------
  0  In     0.499996  0.500005   3.3e-05
  1  In     0.750005  0.249948   0.500033
  2  Cu    -0.000138  0.00018   -0.000157
  3  Cu     0.250046  0.749934   0.499851
  4  Se     0.32851   0.374992   0.250027
  5  Se     0.921433  0.874905   0.249837
  6  Se     0.124984  0.671439   0.750068
  7  Se     0.625163  0.078597   0.750309


In [10]:
builder = SimpleVacancyWorkChain.get_builder()

upd = VaspRelaxUpdater(builder = builder.relax, ).apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'ncore':8 , 'kpar': 4, 'isym': 0, 'symprec': 1e-9}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600*12, queue_name='xhhctdnormal')
upd.set_label(f'{form} RELAX')
upd.set_relax_settings(algo='rd')
# Assign the elemental structures
builder.elemental_structures = {symbols[i]: elemental_struct_path[symbols[i] + '_bulk'].node 
                                for i in range(3)}
assert None not in builder.elemental_structures.values()
builder.supercell_dim = orm.List([2,2,2])  # 222 supercell
# Updated parameters for supercell calculation
builder.supercell_workchain_updates = orm.Dict(
    {'options': {
        'resources': {'tot_num_mpiprocs': 128, 'num_machines':2},
        'max_wallclock_seconds': 3600 * 12, 
        'queue_name': 'xhhctdnormal',
        },
     'incar': {'kpar': 2, 'ncore': 8, 'isym': 0, 'symprec': 1e-9}
    }
)

In [11]:
workpath.show_tree(decorate_by=['exit_status', 'pk'])

workflows 121
├── Ba3PbO_work [0] | 668617
├── Ba3SnO_work [0] | 668797
├── Ca2Pb_work [0] | 669135
├── Ca2Sn_work [0] | 669224
├── Ca3Bi2_work [0] | 669702
├── Ca3PbO_work [0] | 668910
├── GeCdAs2_work [0] | 664271
├── GeCdSb2_work [0] | 664179
├── GeZnBi2_work [0] | 664223
├── GeZnSb2_work [excepted] | 664317
├── GeZnSb2_work_sym_update [0] | 665530
├── InAuSe2_work_sym_update [0] | 666598
├── InCuSe2_work_sym_update [0] | 670540
├── Mg2Pb_work [501] | 670744
├── Mg2Sn_work [501] | 670766
├── Mg3Bi2_work [0] | 669492
├── Mg3PbO_work [0] | 668955
├── PbCdAs2_work [excepted] | 664341
├── PbCdAs2_work_sym_update [0] | 665552
├── PbCdP2_work [excepted] | 664132
├── PbCdP2_work_sym_update [0] | 665574
├── PbZnAs2_work [excepted] | 664157
├── PbZnAs2_work_sym_update [0] | 665596
├── PbZnP2_work [excepted] | 664249
├── PbZnP2_work_sym_update [0] | 665618
├── PbZnSb2_work [0] | 663926
├── SnCdSb2_work [0] | 664110
├── SnZnBi2_work [0] | 664201
├── SnZnSb2_work [excepted] | 664293
├── SnZnSb2